# barexam_qa — Data Cleaning

Run this notebook **before** `EDA_barexam_qa.ipynb`.  
Output: `data/barexam_qa/barexam_qa_clean.parquet`

## 0. Load raw data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

df = pd.read_parquet('./data/barexam_qa/barexam_qa.parquet')
print(f'Shape : {df.shape}')
print(f'Cols  : {df.columns.tolist()}')
df.head(3)

Shape : (1195, 17)
Cols  : ['idx', 'dataset', 'example_id', 'prompt_id', 'source', 'subject', 'question_number', 'prompt', 'question', 'choice_a', 'choice_b', 'choice_c', 'choice_d', 'answer', 'gold_passage', 'gold_idx', 'split']


,idx,dataset,example_id,prompt_id,source,subject,question_number,prompt,question,choice_a,choice_b,choice_c,choice_d,answer,gold_passage,gold_idx,split
0,mbe_0,mbe,0,0,MBE-1972-78-part2,NaN,1,"Paul, the Plaintiff in a personal injury actio...",Paul then called Vic to testify that Dan's car...,admissible because Paul was surprised by Wes's...,admissible because Vic's testimony was relevan...,inadmissible because Paul cannot impeach his o...,inadmissible because Paul is bound by the test...,B,Rules of evidence determine what types of evid...,mbe_0,train
1,mbe_1,mbe,1,0,MBE-1972-78-part2,NaN,2,"Paul, the Plaintiff in a personal injury actio...","On cross-examination of Vic, Dan's attorney as...",admissible to impeach Vic by showing that he h...,admissible to show that Vic is not the kind of...,inadmissible because a witness cannot be impea...,inadmissible because the question of whether V...,D,A matter is considered collateral if “the mat...,mbe_1,train
2,mbe_2,mbe,2,0,MBE-1972-78-part2,NaN,3,"Paul, the Plaintiff in a personal injury actio...",Dan called Zemo as a witness and asked him if ...,objectionable because collateral to the issues...,objectionable because character cannot be prov...,unobjectionable because a foundation for impea...,unobjectionable because Zemo could be expected...,C,"Before a document may be received in evidence,...",mbe_2,train


Note: The dataset we could access from hubspot is the Historical MBE subset — the Barbri subset is not pulicly available (private, held-out test set, not released due to copyright concerns). The full dataset used in 'A Reasoning Focused legal Benchmark' by Zheng et Al. is the combination of both (Historical MBE + Barbri subset). 

## 1. Schema audit
Check dtypes and value ranges for every column.

In [ ]:
print('=== dtypes ===')
print(df.dtypes.to_string())
print()

print('=== numeric columns — value range ===')
num_cols = df.select_dtypes(include='number').columns.tolist()
print(df[num_cols].describe().round(0).to_string())
print()

print('=== string columns — unique value counts ===')
str_cols = df.select_dtypes(include='object').columns.tolist()
for col in str_cols:
    n_unique = df[col].nunique(dropna=True)
    max_len  = df[col].str.len().max()
    print(f'  {col:<20} {n_unique:>5} unique  |  max_len={max_len}')

## 2. Missing values

In [ ]:
missing = (
    df.isnull().sum()
    .rename('n_missing')
    .to_frame()
    .assign(pct=lambda d: (d['n_missing'] / len(df) * 100).round(1))
    .sort_values('n_missing', ascending=False)
)
print(missing.to_string())

# Visual
has_missing = missing[missing['n_missing'] > 0]
if len(has_missing):
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.barh(has_missing.index[::-1], has_missing['pct'].values[::-1], color='#dd8452')
    for i, (col, row) in enumerate(has_missing.iloc[::-1].iterrows()):
        ax.text(row['pct'] + 0.3, i, f"{row['n_missing']} ({row['pct']}%)", va='center', fontsize=9)
    ax.set_xlabel('% missing')
    ax.set_title('Missing values by column')
    ax.set_xlim(0, has_missing['pct'].max() * 1.25)
    plt.tight_layout()
    plt.show()

## 3. Column-by-column inspection

Deep-dive into every column that has issues.

### 3a. `source` — mixed formats + garbage value

In [ ]:
import re

def classify_source(s):
    if pd.isna(s): return 'missing'
    s = str(s).strip()
    if re.fullmatch(r'\d{4}', s):          return 'year'
    if re.fullmatch(r'\d{4}-\w+', s):      return 'year-month'
    if re.fullmatch(r'MBE-[\d\w-]+', s):   return 'MBE-part'
    return 'garbage'

df['_source_type'] = df['source'].apply(classify_source)
print(df['_source_type'].value_counts().to_string())
print()

garbage = df[df['_source_type'] == 'garbage']
if len(garbage):
    print(f'Garbage rows ({len(garbage)}):')
    for _, r in garbage.iterrows():
        print(f"  idx={r['idx']} | split={r['split']} | source (first 120): {str(r['source'])[:120]}…")
    print()
    print('  → Fix: replace with NaN (annotation accidentally stored as source label)')

### 3b. `subject` — 601 missing (~50%)

In [ ]:
print('Missing subject by split:')
print(df.groupby('split')['subject'].apply(lambda s: s.isna().sum()).to_string())
print()
print('Missing subject by source_type:')
print(df.groupby('_source_type')['subject'].apply(lambda s: s.isna().sum()).to_string())
print()
print('Known subject values:')
print(df['subject'].value_counts().to_string())
print()
print('  → subject is missing for ALL MBE-part rows (older exam batches never had subject labels).')
print('  → Imputation options:')
print('     1. Leave as NaN — safest, avoids introducing noise')
print('     2. Predict from question text (e.g. keyword-based heuristic or classifier)')
print('     3. Mark as "Unknown" for analyses that need a non-null category')

### 3c. `prompt` — 750 missing (~63%)

In [ ]:
print('Missing prompt by split:')
print(df.groupby('split')['prompt'].apply(lambda s: s.isna().sum()).to_string())
print()
print('Missing prompt by source_type:')
print(df.groupby('_source_type')['prompt'].apply(lambda s: s.isna().sum()).to_string())
print()
print('  → prompt (fact pattern) is legitimately absent for standalone questions.')
print('     These are NOT data errors — the question is self-contained.')
print('  → No imputation needed. NaN = no fact pattern.')
print('  → For model input: treat missing prompt as empty string (\'\').')

### 3d. `question` — 1 missing

In [ ]:
missing_q = df[df['question'].isna()]
print(f'{len(missing_q)} row(s) with missing question:')
for _, r in missing_q.iterrows():
    print(f"  idx={r['idx']} | source={r['source']} | subject={r['subject']} | split={r['split']}")
    print(f"  prompt   : {str(r['prompt'])[:120]}")
    print(f"  choice_a : {r['choice_a']}")
    print(f"  answer   : {r['answer']}")
    print(f"  gold_passage: {str(r['gold_passage'])[:120]}")
print()
print('  → Row is unrecoverable without original source material.')
print('  → Fix: drop this row.')

### 3e. `choice_d` — 1 missing

In [ ]:
missing_d = df[df['choice_d'].isna()]
print(f'{len(missing_d)} row(s) with missing choice_d:')
for _, r in missing_d.iterrows():
    print(f"  idx={r['idx']} | source={r['source']} | split={r['split']}")
    print(f"  question : {str(r['question'])[:120]}")
    print(f"  choice_a : {r['choice_a']}")
    print(f"  choice_b : {r['choice_b']}")
    print(f"  choice_c : {r['choice_c']}")
    print(f"  choice_d : {r['choice_d']}")
    print(f"  answer   : {r['answer']}")
print()
print('  → If answer != D: question is still usable (just missing one distractor).')
print('  → If answer == D: row is unrecoverable → drop.')
print('  → Fix: drop if answer==D, else keep and note missing choice_d.')

## 4. Apply cleaning

In [ ]:
df_clean = df.copy()
log = []

# 1. source: nullify garbage
mask = df_clean['source'].str.len() > 50
if mask.any():
    log.append(f"source: nullified {mask.sum()} garbage value(s)")
    df_clean.loc[mask, 'source'] = pd.NA

# 2. Drop internal helper column
df_clean.drop(columns=['_source_type'], errors='ignore', inplace=True)

# 3. prompt: NaN → empty string for model input readiness
#    (keep NaN in df_clean for EDA; add a separate model-ready column)
df_clean['prompt_clean'] = df_clean['prompt'].fillna('')
log.append(f"prompt_clean: filled {df_clean['prompt'].isna().sum()} NaN → ''")

# 4. Drop row with missing question (unrecoverable)
missing_q_mask = df_clean['question'].isna()
if missing_q_mask.any():
    log.append(f"Dropped {missing_q_mask.sum()} row(s) with missing question: {df_clean[missing_q_mask]['idx'].tolist()}")
    df_clean = df_clean[~missing_q_mask].reset_index(drop=True)

# 5. Drop row with missing choice_d only if answer == D
bad_d = df_clean[df_clean['choice_d'].isna() & (df_clean['answer'].str.upper() == 'D')]
if len(bad_d):
    log.append(f"Dropped {len(bad_d)} row(s) with missing choice_d where answer=D: {bad_d['idx'].tolist()}")
    df_clean = df_clean.drop(index=bad_d.index).reset_index(drop=True)

print('=== Cleaning log ===')
for entry in log:
    print(f'  • {entry}')
print()
print(f'Shape before : {df.shape}')
print(f'Shape after  : {df_clean.shape}')

## 5. Post-cleaning check

In [ ]:
print('=== Remaining missing values ===')
remaining = df_clean.isnull().sum()
remaining = remaining[remaining > 0]
if len(remaining):
    print(remaining.to_string())
    print()
    print('  subject (601): expected — MBE-part rows have no subject label')
    print('  prompt  (750): expected — standalone questions have no fact pattern')
    print('  choice_d (n) : expected if answer != D (non-critical missing)')
else:
    print('None')

print()
print('=== answer column sanity check ===')
print(df_clean['answer'].str.upper().value_counts().sort_index().to_string())
unexpected = df_clean[~df_clean['answer'].str.upper().isin(['A','B','C','D'])]
if len(unexpected):
    print(f'  ⚠️  {len(unexpected)} unexpected answer value(s):')
    print(unexpected[['idx','answer']].to_string())
else:
    print('  ✓ All answers are A/B/C/D')

## 6. Save

In [ ]:
OUT = './data/barexam_qa/barexam_qa_clean.parquet'
df_clean.to_parquet(OUT, index=False)
print(f'Saved → {OUT}')
print(f'Final shape: {df_clean.shape}')